# Transform NKR Excel export into NocoDB CSV

This Notebook helps with importing data by Normenkontrollrat to our own database, running NocoDB.

Install the dependencies by running `uv sync` in this directory.

When running the cells, check whether the column names are correct, as they have sometimes changed between different exports.

In [ ]:
import pandas as pd
import numpy as np

# Change this path as required
input_path = "~/Downloads/260205_DB_Auszug_Digitalcheck.xlsx"
df = pd.read_excel(input_path)
df

In [ ]:
# NKRNr	Regelungsart	Ressort	Titel	EingangVP	EingangED	Erledigungsdatum	VP1_ITLösung	VP2_Verpflichtungen	VP3_Datenaustausch	VP4_Interaktion	VP5_Automatisierung	ED1_Kommunikation	ED2_Daten	ED3_Datenschutz	ED4_Klarheit	ED5_Automatisierung	EDDarstellung
out = pd.DataFrame()
out["NKRNr"] = df["A_NKR-Nr"]
out["Regelungsart"] = df["Regelungsart"]
out["Ressort"] = df["Ressort"]
out["Titel"] = df["A_Titel"]
out["EingangVP"] = df["I_EingangVP"]
out["EingangED"] = df["I_EingangED"]
out["Erledigungsdatum"] = df["B_Erledigungsdatum"]

out

In [ ]:
# copy TRUE/FALSE fields
vp_map = {
    "VP1_ITLösung": "I_VPITLösung",
    "VP2_Verpflichtungen": "I_VPVerpflichtungen",
    "VP3_Datenaustausch": "I_VPDatenaustausch",
    "VP4_Interaktion": "I_VPInteraktion",
    "VP5_Automatisierung": "I_VPAutomatisierung",
}

for out_col, source_col in vp_map.items():
    assert set(df[source_col].unique()).issubset({True, False})
    out[out_col] = df[source_col]

out

In [ ]:
df.columns

In [ ]:
# explore matching columns
[col for col in df.columns.to_list() if "tech" in col.lower()]

In [ ]:
# copy Ja/Nein/Teilweise/Nicht relevant fields
expected_values = {"Ja", "Nein", "Teilweise", "Nicht relevant", np.nan}

ed_map = {"ED1_Kommunikation": "I_EDKommunikationMA",
          "ED2_Daten": "I_EDDatenMA",
          "ED3_Datenschutz": "I_EDDatenschutzMA",
          "ED4_Klarheit": "I_EDKlarheitMA",
          "ED5_Automatisierung": "I_EDAutomatisierungMA",
          # v2 principles
          "EDv2_1_Angebote": "I_EDAngeboteMA",
          "EDv2_2_Wiederverwendung": "I_EDWiederverwendungMA",
          "EDv2_3_Technologien": "I_EDTechnologienMA",
          "EDv2_4_Automatisierung": "I_EDAutomatisierungNeuMA",
          "EDv2_5_Datenschutz": "I_EDDatenschutzNeuMA",
          }


for out_col, source_col in ed_map.items():

    assert set(df[source_col].unique()).issubset(
        expected_values), f"Unexpected items {set(df[source_col].unique()).difference(expected_values)} column {source_col}"
    out[out_col] = df[source_col]
out

In [ ]:
df["I_EDDatenschutzMA"].fillna(np.nan).count()

In [ ]:
out = out.drop_duplicates()

In [ ]:
date_cols = ["EingangVP", "EingangED", "Erledigungsdatum"]

for col in date_cols:
    out[col] = out[col].dt.strftime('%Y-%m-%d')

In [ ]:
# Alternative A: Save to CSV and load into NocoDB manually
out.to_csv(input_path.replace(".xlsx", ".csv"), index=False)

In [ ]:
# Alternative B: Upload to NocoDB via API

# read NocoDB token from 1Password
import subprocess
nocodb_token = subprocess.run(
    ["op", "read", "op://Employee/NOCODB token/credential"],
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

In [ ]:
# check what records are already in the table
records = []

import requests
is_last_page = False
offset = 0
while not is_last_page:
    url = f"https://metrics.ds4g.dev:38081/api/v2/tables/mhsmsi0pu31n3xw/records?offset={offset}"
    print(url)
    res = requests.get(url, headers={"xc-token": nocodb_token})
    j = res.json()
    pageinfo = j["pageInfo"]
    offset += pageinfo["pageSize"]
    is_last_page = pageinfo["isLastPage"]
    records.extend(j["list"])
len(records)


In [ ]:
nkr_nrs = {record["NKRNr"] for record in records}
nkr_nrs

In [ ]:
set(out.columns.to_list()).difference(records[0].keys())

In [ ]:
set(records[0].keys()).difference(out.columns.to_list())


In [ ]:
# check whether any Digitalchecks are already in NocoDB
to_ignore = {nr for nr in out["NKRNr"].to_list() if nr in nkr_nrs}

print(f"Will ignore {len(to_ignore)} records that are already in NocoDB: {to_ignore}")
# replace nan with None
payload = out.replace({np.nan: None}).to_dict(orient="records")
payload = [item for item in payload if item["NKRNr"] not in to_ignore]
payload

In [ ]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

url = f"{os.getenv('NOCODB_URL')}/api/v2/tables/mhsmsi0pu31n3xw/records"

successful_items = []

for item in payload:
    print(item["NKRNr"], end=": ")
    res = requests.post(url, headers={"xc-token": nocodb_token}, json=item)
    res.raise_for_status()
    print(res.status_code)
    successful_items.append(item["NKRNr"])